# ARYA RAG: Document Contract

**Branch:** `feature/rag-document-contract`

This notebook defines the **contracts only**: enums, DTOs, error types, and the function signatures (ports) for RAG. There is no PDF parsing, chunking, embedding, SQL, or FastAPI code here.

**Why:** Person 2 (database) implements `RagRepository`, Person 3 (agent) calls `RagService`, and you build the real implementation in later branches. All of you code against these shapes.

**Rules baked into the contract**
- `owner_id` is the verified Clerk ID, always a `str`, never a `UUID`.
- Every repository method takes `owner_id` first, so ownership scoping is part of the contract.
- "No results" is an empty list. "Something broke" is an exception.
- Citations only come from retrieved chunks.

**Run order:** install, enums, DTOs, errors, ports, fake repository, tests, then copy cells into `.py` files.

## 0. Install

In [2]:
# Only pydantic is needed (FastAPI already depends on it)
%pip install -q "pydantic>=2"

Note: you may need to restart the kernel to use updated packages.


## 1. Enums: status, file type, error codes

In [3]:
from __future__ import annotations

from datetime import datetime
from enum import Enum
from typing import BinaryIO, Protocol, runtime_checkable
from uuid import UUID

from pydantic import BaseModel, Field


class DocumentStatus(str, Enum):
    """Lifecycle: uploaded -> processing -> ready | failed.
    A document is searchable ONLY when status == READY."""
    UPLOADED = "uploaded"
    PROCESSING = "processing"
    READY = "ready"
    FAILED = "failed"


class FileType(str, Enum):
    PDF = "pdf"
    TXT = "txt"
    MD = "md"


class IngestionErrorCode(str, Enum):
    UNSUPPORTED_FILE_TYPE = "UNSUPPORTED_FILE_TYPE"
    FILE_TOO_LARGE = "FILE_TOO_LARGE"
    NO_EXTRACTABLE_TEXT = "NO_EXTRACTABLE_TEXT"   # scanned PDF, OCR not supported yet
    EMBEDDING_FAILED = "EMBEDDING_FAILED"
    INDEXING_FAILED = "INDEXING_FAILED"
    DUPLICATE_DOCUMENT = "DUPLICATE_DOCUMENT"

## 2. DTOs

In [4]:
class DocumentRecord(BaseModel):
    """Document metadata returned by the repository and the documents API."""
    id: UUID
    owner_id: str                       # Clerk user ID (string)
    filename: str
    file_type: FileType
    content_sha256: str
    status: DocumentStatus
    chunk_count: int = 0
    error_code: IngestionErrorCode | None = None
    error_message: str | None = None    # safe text only, no stack traces or file content
    created_at: datetime
    updated_at: datetime


class IngestionResult(BaseModel):
    document_id: UUID
    status: DocumentStatus
    chunk_count: int = 0
    error_code: IngestionErrorCode | None = None
    error_message: str | None = None


class ChunkRecord(BaseModel):
    """One chunk to be stored by the repository (written during ingestion)."""
    chunk_index: int = Field(ge=0)
    text: str
    token_count: int = Field(ge=0)
    page: int | None = None
    section: str | None = None
    embedding: list[float]


class RetrievedChunk(BaseModel):
    """One chunk returned by retrieval (read during question answering)."""
    chunk_id: UUID
    document_id: UUID
    title: str                          # document filename
    text: str
    page: int | None = None
    section: str | None = None
    score: float                        # similarity, higher is better


class SourceCitation(BaseModel):
    """What the chat API returns to the UI. Built ONLY from a RetrievedChunk."""
    document_id: UUID
    title: str
    chunk_id: UUID
    excerpt: str
    page: int | None = None

    @classmethod
    def from_chunk(cls, chunk: RetrievedChunk, max_chars: int = 300) -> "SourceCitation":
        return cls(
            document_id=chunk.document_id,
            title=chunk.title,
            chunk_id=chunk.chunk_id,
            excerpt=chunk.text[:max_chars],
            page=chunk.page,
        )


class RetrievalFilters(BaseModel):
    document_ids: list[UUID] | None = None   # narrow WITHIN the user's own documents
    top_k: int = Field(default=5, ge=1, le=20)
    min_score: float | None = None

## 3. Errors

In [5]:
class RagError(Exception):
    """Base class for RAG failures. Never used for 'no results'."""


class IngestionError(RagError):
    def __init__(self, code: IngestionErrorCode, message: str):
        super().__init__(message)
        self.code = code
        self.message = message


class RetrievalError(RagError):
    """Embedding or database failure during retrieval."""


class DocumentNotFoundError(RagError):
    """Document does not exist OR is not owned by the caller (do not reveal which)."""

## 4. Ports (signatures only)

- `RagService`: what **Person 3 (agent)** and the API call.
- `Embedder`: wrapper around the embedding model; easy to mock.
- `RagRepository`: implemented by **Person 2**. RAG code never writes SQL.

In [7]:
@runtime_checkable
class Embedder(Protocol):
    dimension: int   # must match the pgvector column, e.g. 384 for all-MiniLM-L6-v2

    def embed_texts(self, texts: list[str]) -> list[list[float]]: ...
    def embed_query(self, text: str) -> list[float]: ...


@runtime_checkable
class RagRepository(Protocol):
    def create_document(self, owner_id: str, filename: str, file_type: FileType,
                        content_sha256: str, storage_key: str) -> DocumentRecord: ...

    def find_by_hash(self, owner_id: str, content_sha256: str) -> DocumentRecord | None: ...

    def set_status(self, owner_id: str, document_id: UUID, status: DocumentStatus,
                   error_code: IngestionErrorCode | None = None,
                   error_message: str | None = None) -> None: ...

    def replace_chunks(self, owner_id: str, document_id: UUID,
                       chunks: list[ChunkRecord]) -> None:
        """ONE transaction: delete old chunks, insert new ones, set status READY
        and chunk_count. Re-ingestion must leave no stale chunks."""
        ...

    def search_chunks(self, owner_id: str, query_embedding: list[float],
                      filters: RetrievalFilters) -> list[RetrievedChunk]:
        """Only chunks of READY documents owned by owner_id."""
        ...

    def list_documents(self, owner_id: str) -> list[DocumentRecord]: ...

    def get_document(self, owner_id: str, document_id: UUID) -> DocumentRecord | None: ...

    def delete_document(self, owner_id: str, document_id: UUID) -> bool: ...


@runtime_checkable
class RagService(Protocol):
    def ingest_document(self, user_id: str, document_id: UUID,
                        file: BinaryIO, filename: str) -> IngestionResult:
        """Raises IngestionError for expected failures."""
        ...

    def retrieve(self, user_id: str, query: str,
                 filters: RetrievalFilters | None = None) -> list[RetrievedChunk]:
        """Returns [] when nothing matches. Raises RetrievalError on failure."""
        ...

## 5. In-memory fake repository

Proves the contract is implementable and gives Persons 1 and 3 something to test against before the real database exists. It uses plain cosine similarity, no database.

In [9]:
import math
from datetime import timezone
from uuid import uuid4


def _now() -> datetime:
    return datetime.now(timezone.utc)


def _cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0


class InMemoryRagRepository:
    def __init__(self):
        self.docs: dict[UUID, DocumentRecord] = {}
        self.chunks: dict[UUID, list[tuple[UUID, ChunkRecord]]] = {}

    def create_document(self, owner_id, filename, file_type, content_sha256, storage_key):
        if self.find_by_hash(owner_id, content_sha256):
            raise IngestionError(IngestionErrorCode.DUPLICATE_DOCUMENT, "Document already uploaded.")
        rec = DocumentRecord(
            id=uuid4(), owner_id=owner_id, filename=filename, file_type=file_type,
            content_sha256=content_sha256, status=DocumentStatus.UPLOADED,
            created_at=_now(), updated_at=_now(),
        )
        self.docs[rec.id] = rec
        return rec

    def find_by_hash(self, owner_id, content_sha256):
        return next((d for d in self.docs.values()
                     if d.owner_id == owner_id and d.content_sha256 == content_sha256), None)

    def _owned(self, owner_id, document_id):
        d = self.docs.get(document_id)
        return d if d and d.owner_id == owner_id else None

    def set_status(self, owner_id, document_id, status, error_code=None, error_message=None):
        d = self._owned(owner_id, document_id)
        if not d:
            raise DocumentNotFoundError()
        d.status, d.error_code, d.error_message, d.updated_at = status, error_code, error_message, _now()

    def replace_chunks(self, owner_id, document_id, chunks):
        d = self._owned(owner_id, document_id)
        if not d:
            raise DocumentNotFoundError()
        self.chunks[document_id] = [(uuid4(), c) for c in chunks]   # replaces old chunks
        d.status, d.chunk_count, d.updated_at = DocumentStatus.READY, len(chunks), _now()
        d.error_code = d.error_message = None

    def search_chunks(self, owner_id, query_embedding, filters):
        hits = []
        for d in self.docs.values():
            if d.owner_id != owner_id or d.status != DocumentStatus.READY:
                continue
            if filters.document_ids and d.id not in filters.document_ids:
                continue
            for cid, c in self.chunks.get(d.id, []):
                s = _cosine(query_embedding, c.embedding)
                if filters.min_score is None or s >= filters.min_score:
                    hits.append(RetrievedChunk(chunk_id=cid, document_id=d.id, title=d.filename,
                                               text=c.text, page=c.page, section=c.section, score=s))
        return sorted(hits, key=lambda h: h.score, reverse=True)[: filters.top_k]

    def list_documents(self, owner_id):
        return [d for d in self.docs.values() if d.owner_id == owner_id]

    def get_document(self, owner_id, document_id):
        return self._owned(owner_id, document_id)

    def delete_document(self, owner_id, document_id):
        if not self._owned(owner_id, document_id):
            return False
        del self.docs[document_id]
        self.chunks.pop(document_id, None)
        return True


assert isinstance(InMemoryRagRepository(), RagRepository)
print("Fake repository satisfies RagRepository")

Fake repository satisfies RagRepository


## 6. Contract tests

In [10]:

def _raises(exc, fn):
    try:
        fn()
    except exc:
        return True
    return False


# --- enums match the project guide
assert [s.value for s in DocumentStatus] == ["uploaded", "processing", "ready", "failed"]
assert {t.value for t in FileType} == {"pdf", "txt", "md"}

# --- owner_id is a plain string (Clerk ID), not a UUID
rec = DocumentRecord(id=uuid4(), owner_id="user_2abc123XYZ", filename="ML_Unit_1.pdf",
                     file_type=FileType.PDF, content_sha256="a" * 64,
                     status=DocumentStatus.UPLOADED, created_at=_now(), updated_at=_now())
assert rec.owner_id == "user_2abc123XYZ"

# --- filter bounds
from pydantic import ValidationError
assert _raises(ValidationError, lambda: RetrievalFilters(top_k=0))
assert _raises(ValidationError, lambda: RetrievalFilters(top_k=100))
assert RetrievalFilters().top_k == 5

# --- repository behaviour
repo = InMemoryRagRepository()
alice, bob = "user_alice", "user_bob"

doc = repo.create_document(alice, "svm.md", FileType.MD, "h1", "storage/svm.md")
repo.set_status(alice, doc.id, DocumentStatus.PROCESSING)
repo.replace_chunks(alice, doc.id, [
    ChunkRecord(chunk_index=0, text="SVM finds a hyperplane", token_count=5, page=1, embedding=[1.0, 0.0]),
    ChunkRecord(chunk_index=1, text="Kernels handle nonlinear data", token_count=5, page=2, embedding=[0.0, 1.0]),
])
assert repo.get_document(alice, doc.id).status == DocumentStatus.READY
assert repo.get_document(alice, doc.id).chunk_count == 2

# retrieval returns the best match with citation metadata
hits = repo.search_chunks(alice, [1.0, 0.1], RetrievalFilters(top_k=1))
assert hits[0].text.startswith("SVM") and hits[0].page == 1
cite = SourceCitation.from_chunk(hits[0])
assert cite.chunk_id == hits[0].chunk_id

# ownership isolation: Bob sees nothing and cannot read or delete Alice's document
assert repo.search_chunks(bob, [1.0, 0.1], RetrievalFilters()) == []
assert repo.get_document(bob, doc.id) is None
assert repo.delete_document(bob, doc.id) is False

# non-ready documents are not searchable
doc2 = repo.create_document(alice, "draft.md", FileType.MD, "h2", "storage/draft.md")
assert all(h.document_id != doc2.id for h in repo.search_chunks(alice, [1.0, 0.0], RetrievalFilters()))

# duplicate upload by the same user is rejected, but another user may upload the same file
assert _raises(IngestionError, lambda: repo.create_document(alice, "svm.md", FileType.MD, "h1", "x"))
repo.create_document(bob, "svm.md", FileType.MD, "h1", "storage/bob_svm.md")

# re-ingestion leaves no stale chunks
repo.replace_chunks(alice, doc.id, [
    ChunkRecord(chunk_index=0, text="New content", token_count=2, embedding=[1.0, 0.0])])
assert repo.get_document(alice, doc.id).chunk_count == 1
assert len(repo.search_chunks(alice, [1.0, 0.0], RetrievalFilters(top_k=20))) == 1

# delete removes document and chunks
assert repo.delete_document(alice, doc.id) is True
assert doc.id not in repo.chunks

print("All contract tests passed")

All contract tests passed


## 7. Move this into the repository

Copy cells into these files (keep the import block at the top of each):

| Notebook cells | Target file |
|---|---|
| 1 to 3 (enums, DTOs, errors) | `backend/app/rag/contracts.py` |
| 4 (ports) | `backend/app/rag/ports.py`, importing from `contracts` |
| 5 (fake repository) | `backend/tests/fakes/in_memory_rag_repository.py` |
| 6 (tests) | `backend/tests/unit/rag/test_contracts.py`, rewritten as `pytest` functions |

**Before opening the PR**
1. Add a short "RAG contract" section to the project guide pointing to these files.
2. Send `RagRepository` to Person 2 and `RagService` plus `RetrievedChunk` to Person 3 for review.
3. Commit in small steps and open the PR against your integration branch.

**Not in this branch:** extraction, chunking, embedding, SQL or pgvector, FastAPI routes.